In [ ]:
!pip install -q chromadb sentence-transformers datasets

In [ ]:
import os
import time
import json
import chromadb
import gradio as gr
from openai import OpenAI
from datasets import load_dataset
from google.colab import userdata
from IPython.display import Markdown, display
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

openai = OpenAI(api_key=OPENAI_API_KEY)
db_name = "faqs_qus_ans"
AI_MODEL = "gpt-5-nano"
chroma_client = chromadb.Client()
# chromadb.PersistentClient(path=db_name)
EMBEDDING_MODEL= "sentence-transformers/all-MiniLM-L6-v2"

In [ ]:
faqs_embeddings = SentenceTransformer(EMBEDDING_MODEL)

In [ ]:
collection = chroma_client.get_or_create_collection(name=db_name)

If you need to know *where* the `datasets` library caches the data locally, you can find that by checking the `datasets` cache directory, but this is usually an internal detail not directly tied to a 'path' of the loaded dataset object itself.

In [ ]:
faqs_dataset = load_dataset("NebulaByte/E-Commerce_FAQs")

In [ ]:
list_of_data = faqs_dataset["train"]

In [ ]:
# Prepare empty arrays for bulk insertion
documents = []
metadatas = []
ids = []


# Process and sanitize data strictly
for i, item in enumerate(list_of_data):
    # Fallback to loop index string if question_id is missing or invalid
    doc_id = str(item.get('question_id', f"id_{i}"))

    # Text payload used for embedding vector calculation
    # doc_text = item.get('que_ans', '')

    doc_text = f"""
    Question:
    {item['question']}

    Answer:
      {item['answer']}

    Category:
      {item['category']}

    Parent Category:
      {item['parent_category']}

    URL:
      {item['faq_url']}
    """

    # Construct a flat metadata dictionary matching Chroma's specifications
    raw_metadata = {
        'parent_category': item.get('parent_category'),
        'category_id': item.get('category_id'),
        'category': item.get('category'),
        'question': item.get('question'),
        'answer': item.get('answer'),
        'faq_url': item.get('faq_url')
    }

    # Strictly sanitize keys and values to match ChromaDB MetadataValue spec
    clean_metadata = {}
    for key, val in raw_metadata.items():
        if val is None:
            clean_metadata[key] = ""  # Convert None to empty string
        elif isinstance(val, (str, int, float, bool)):
            clean_metadata[key] = val  # Keep allowed primitive data types
        elif isinstance(val, (dict, list)):
            clean_metadata[key] = json.dumps(val)  # Flatten complex types to valid strings
        else:
            clean_metadata[key] = str(val)  # Cast any remaining types safely

    # Append validated structures to final collections
    documents.append(doc_text)
    ids.append(doc_id)
    metadatas.append(clean_metadata)

# 3. Safe, batch insertion into database
if collection.count() == 0:
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )
else:
    print("Collection already contains data. Skipping insertion.")

In [ ]:
count = collection.count()
embedded_faqs = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
# sample_embedding
dimensions = len(embedded_faqs)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

In [ ]:
SYSTEM_PREFIX = """
You are a helpful E-Commerce FAQ Assistant.

Your task is to answer user questions using only the provided FAQ data.

Instructions:

* Carefully analyze the provided FAQ information.
* Answer the user's question using the FAQ content.
* Keep responses realistic, concise, and easy to understand.
* Use a natural, customer-support style tone.
* Do not use excessive bullet points or lengthy explanations.
* Do not invent, assume, or hallucinate information.
* If the answer cannot be found in the provided FAQ data, respond with "No".
* Prefer the FAQ answer field over any other information.
* Include the source URL when available.

The provided FAQ data may contain:

* question
* answer
* category
* parent_category
* faq_url
* additional metadata


Your goal is to provide trustworthy, context-based responses that accurately reflect the available FAQ information.

Context:
{context}
"""

In [ ]:
def retrieve_documents(question):
  results = collection.query(
    query_texts=[question],
    n_results=3,
    include=["documents" , "metadatas" , "distances"]
 )
  context = "\n\n".join(results["documents"][0])
  metadatas = "\n\n".join(results["metadatas"][0][0])
  best_distance = results["distances"][0][0]
  if best_distance > 1.0:
    return {
      "found": False,
      "context": "",
      "message": "No relevant FAQ found."
    }
  return {
    "found": True,
    "context" : context,
    "distance" : best_distance,
    "metadatas" : metadatas
  }

In [ ]:
def build_prompt(context):
  system_prompt = SYSTEM_PREFIX.format(context=context)
  return system_prompt

In [ ]:
def chat(question , history):

  retrieval = retrieve_documents(question)
  if not retrieval["found"]:
    yield retrieval["message"]
    return

  system_prompt = build_prompt(retrieval)

  messages = [
    {"role": "system", "content": system_prompt}
  ]

  for user_msg, assistant_msg in history:
    messages.append({"role": "user", "content": user_msg})
    messages.append({"role": "assistant", "content": assistant_msg})


  messages.append({"role": "user", "content": question})

  ai_response = openai.chat.completions.create(
        model=AI_MODEL,
        messages=messages,
        stream=True
  )
  response = ""

  for chunk in ai_response:
      if chunk.choices:
          delta = chunk.choices[0].delta.content
          if delta:
              response += delta
              yield response

In [ ]:
for chunk in chat("hi", []):
    print(chunk)

In [ ]:
gr.ChatInterface(fn=chat).launch()